<div dir="rtl">
<h1>میان‌بر دوم به کجا وصل می‌شود؟</h1>
<p>درس 46 از 76 · یک بلوک را از قطعه‌های آشنا بسازیم · <code dir="ltr">40-block</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html">📖 بازگشت به همین درس</a></p>
<p>Pre-Norm واقعی را بازسازی کنید و خطایی را بگیرید که Shape را عوض نمی‌کند.</p><p>پیش‌نیاز: Attention چندسر، FFN، Residual و Layer Normalization را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر Attention اصلاحی به x اضافه کند، FFN باید کدام نمایش را بگیرد و جمع دوم باید با کدام مقدار انجام شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
block = TransformerBlock(ModelConfig(12,8,8,2,1,0.)).eval()
x = torch.randn(2,5,8)
trace = {}
with torch.no_grad():
    expected = block(x,trace=trace)
print('block input/output:',x.shape,expected.shape)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع manual_block(block,x) زوج (after_attention, output) برگرداند. زیرلایه‌های خود block را به کار ببرید، ولی block(x) را صدا نزنید. ترتیب Pre-Norm و هر دو Residual را حفظ کنید.</p>
</div>

In [ ]:
def manual_block(block, x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = manual_block(block,x)
    if result is None: return False
    after_attention,out = result
    torch.testing.assert_close(out,expected)
    torch.testing.assert_close(after_attention,x+block.attention(block.norm_1(x)))
    torch.testing.assert_close(out-after_attention,trace['feed_forward'])
    y = torch.randn(1,3,8)
    torch.testing.assert_close(manual_block(block,y)[1],block(y))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط residual را روی همان بلوک خاموش کنید. در این API هر دو جمع حذف می‌شوند؛ Layer Normalization و Mask باقی می‌مانند. از اختلاف عددی، حکم کیفیت آموزش صادر نکنید.</p>
</div>

In [ ]:
with torch.no_grad():
    without = block(x,residual=False)
    manual_without = block.feed_forward(block.norm_2(block.attention(block.norm_1(x))))
    torch.testing.assert_close(without,manual_without)
    print('same shape:',without.shape,'difference:',(without-expected).abs().mean().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>نسخهٔ خراب جمع دوم را دوباره به ورودی آغاز بلوک وصل می‌کند. تابع repair_block(block,x) فقط خروجی نهایی صحیح را برگرداند.</p>
</div>

In [ ]:
with torch.no_grad():
    middle = x+block.attention(block.norm_1(x))
    wrong = x+block.feed_forward(block.norm_2(middle))
print('same shape:',wrong.shape,'wrong skip difference:',(wrong-expected).abs().max().item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_block(block, x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_block(block,x)
    if result is None: return False
    torch.testing.assert_close(result,expected)
    for T in (1,4):
        y = torch.randn(1,T,8)
        torch.testing.assert_close(repair_block(block,y),block(y))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>TransformerBlock واقعی هم در v5 و هم در MiniGPT استفاده می‌شود. trace['feed_forward'] اصلاح FFN پیش از جمع است، نه خروجی نهایی بلوک؛ این تفاوت در آزمون میانی سنجیده شد.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام مقایسه نشان داد بلوک فقط هم‌شکل نیست، بلکه همان محاسبهٔ مرجع را انجام می‌دهد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/40-block.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>